# Epstein Files Analysis

For the purpose of this project, I will be restricting my data purely to email documents.

## Identifying Emails

In [36]:
import re
import glob
from collections import defaultdict
import pandas as pd

In [14]:
DATA_FILES = glob.glob("../docs/TEXT/**/*.txt")

### Email Header Validation

In [15]:
REQUIRED_HEADERS = ["From:", "To:", "Sent:"]

'''
Checks if a given document is an email by checking for required headers

scan_limit used to reduce the # of lines to search for headers
'''
def isDocEmail(text: list[str], scan_limit: int = 20):
    reduced_text = text[:scan_limit]
    return all(
        any(line.strip().startswith(header) for line in reduced_text)
        for header in REQUIRED_HEADERS
    )

### Labeling Email Documents

In [ ]:
EMAIL_DOCS = set()

'''
From the global DATA_FILES, find documents that are emails

Email documents are any document that contain the REQUIRED_HEADERS
'''
def findEmails():
    for file in DATA_FILES:
        with open(file, "rt", encoding="utf8") as f:
            text = f.readlines()

            if (isDocEmail(text)):
                EMAIL_DOCS.add(file)

findEmails()

### Extracting Email information

In [92]:
OPTIONAL_HEADERS = ["Subject:", "Importance:", "Attachments:"]

'''
Given a filepath to an email document, parse the email and
extract headers + content
'''
def parseEmail(filepath: str) -> defaultdict:
    with open(filepath, "rt", encoding="utf-8") as f:
        text : list[str] = f.readlines()

    data = defaultdict(str)
    last_header_loc = 0

    # Keep track of seen headers
    # Email chains wont overwrite top most header information
    seen = {h : False for h in REQUIRED_HEADERS + OPTIONAL_HEADERS}

    for i, line in enumerate(text):
        strp_line = line.strip()

        # Get all header information
        for header in REQUIRED_HEADERS + OPTIONAL_HEADERS:
            if (strp_line.startswith(header) and (not seen[header])):
                data[header[:-1]] = strp_line.removeprefix(header).strip()
                last_header_loc = max(last_header_loc, i)
                seen[header] = True
        
        # If we've seen all possible header information, stop
        if all(v for v in seen.values()):
            break
    
    body = "".join(text[last_header_loc + 1:]).strip()

    data["filename"] = filepath.split("\\")[-1]
    data["filepath"] = filepath
    data["body"] = body

    return data

### Parse Emails Into Dataframe

In [93]:
COLS = ["filename", "filepath", "body"].extend(REQUIRED_HEADERS + OPTIONAL_HEADERS)

parsed_emails = []

for doc in EMAIL_DOCS:
    data : dict = dict(parseEmail(doc))

    parsed_emails.append(data)

emails = pd.DataFrame(parsed_emails, columns=COLS)

In [97]:
# emails.to_parquet("parsed_emails.parquet")
emails

,From,Sent,To,Subject,Importance,filename,filepath,body,Attachments
0,Katherine Keating,10/28/2011 4:33:44 PM,jeffrey epstein [jeevacation@gmail.com],Re: Keating Interview,High,HOUSE_OVERSIGHT_029553.txt,../docs/TEXT\001\HOUSE_OVERSIGHT_029553.txt,Yes. Just arrived back from San Fran. \nOn Oct...,NaN
1,,5/8/2017 3:26:51 PM,jeevacation@gmail.com,Re:,High,HOUSE_OVERSIGHT_032531.txt,../docs/TEXT\002\HOUSE_OVERSIGHT_032531.txt,a 2 timer? \nIn a message dated 5/8/2017 11:19...,NaN
2,Linda Stone,8/12/2015 5:22:10 AM,Jeffrey Epstein [jeeyacation@gmail.com],Please,High,HOUSE_OVERSIGHT_032220.txt,../docs/TEXT\002\HOUSE_OVERSIGHT_032220.txt,Please tell me Trump will not win the election...,NaN
3,Kathy Ruemmler,9/21/2014 2:54:20 AM,jeffrey E. [jeeyacation@gmail.com],Re: Floor Statement of Senator Barack Obama on...,High,HOUSE_OVERSIGHT_030123.txt,../docs/TEXT\001\HOUSE_OVERSIGHT_030123.txt,"Yep. Will be there. \nOn Sep 20, 2014 7:54 PM,...",NaN
4,,,,,High,HOUSE_OVERSIGHT_032785.txt,../docs/TEXT\002\HOUSE_OVERSIGHT_032785.txt,Just tried u. Am at 6178179452 \nSent from my ...,NaN
...,...,...,...,...,...,...,...,...,...
2142,Tonja Haddad Coleman_____________________________,10/18/2013 10:38:16 PM,Darren Indyke Jeffrey Epstein [jeevacation@gma...,Fwd: Request for comment from The Mail on Sund...,NaN,HOUSE_OVERSIGHT_030624.txt,../docs/TEXT\001\HOUSE_OVERSIGHT_030624.txt,"Jeffrey Epstein \nTonja Haddad Coleman, Esq. \...",NaN
2143,,9/9/2017 3:40:30 PM,jeffrey E. [jeeyacation@gmail.com],Re:,High,HOUSE_OVERSIGHT_032585.txt,../docs/TEXT\002\HOUSE_OVERSIGHT_032585.txt,what is your shoes size if you dont mind me as...,NaN
2144,,,,,High,HOUSE_OVERSIGHT_032591.txt,../docs/TEXT\002\HOUSE_OVERSIGHT_032591.txt,very well .. plz when you have a minute send m...,NaN
2145,Katherine Keating,10/31/2011 8:21:27 AM,Katherine Keating,After Words,NaN,HOUSE_OVERSIGHT_029558.txt,../docs/TEXT\001\HOUSE_OVERSIGHT_029558.txt,"Dear Friends, \nI wanted to share with you the...",NaN


# Cleaning Data

Now that the data has been parsed, we have to clean it.

For example, dates are in multiple formats. Info has been redacted, so it is missing. etc.

### Turning Dates Into Timestamps

In [100]:
pd.to_datetime(emails["Sent"].str.replace(r"\s*\(UTC\)", "", regex=True))

C:\Users\Connor\AppData\Local\Temp\ipykernel_84404\1974472875.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(emails["Sent"].str.replace(r"\s*\(UTC\)", "", regex=True))


DateParseError: Unknown datetime string format, unable to parse: 0/126/r2013 4:17 PM, at position 768